# Critical-but-Fragile: Mapping Fragility Across Functional Categories of Open Source Infrastructure

**ECP77594 — Computational Methods in Economics — Final Project**

Methods from the module:
- **Text as data** (Stage 3): NMF topic modelling on package descriptions
- **Penalised regression** (Stage 4c): LASSO cross-validated validation of fragility weights
- **Monte Carlo simulation** (Stage 7): Bootstrap confidence intervals

## Setup and Configuration

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score

# Paths — data/ holds analysis-ready CSVs produced by preprocess.py.
# Run `python preprocess.py` once before executing this notebook.
PROJECT_DIR = os.getcwd()
DATA_DIR = os.path.join(PROJECT_DIR, "data")
PACKAGES_FILE = os.path.join(DATA_DIR, "packages_loadbearing.csv")
EUROSTAT_FILE = os.path.join(DATA_DIR, "eurostat_gva_2022.csv")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_PLATFORMS = ["NPM", "Maven", "Pypi", "Rubygems", "NuGet", "Packagist", "Cargo"]
LOAD_BEARING_THRESHOLD = 100
RISK_PERCENTILE = 75
SNAPSHOT_DATE = datetime(2020, 1, 12)

W_CONTRIBUTOR, W_STALENESS, W_ISSUES, W_BUSFACTOR = 0.30, 0.30, 0.20, 0.20
W_DEP_PKG, W_DEP_REPO = 0.65, 0.35

N_TOPICS = 25
N_BOOTSTRAP = 1000
RANDOM_SEED = 42

# Human-readable topic labels, assigned after inspecting auto-generated
# NMF labels and the actual package composition of each cluster.
HUMAN_TOPIC_LABELS = {
    0:  "Generic/legacy infrastructure (residual)",
    1:  "React ecosystem",
    2:  "JS utility libraries",
    3:  "Application frameworks (.NET/PHP)",
    4:  "Build tooling (bundlers)",
    5:  "Cloud & API clients",
    6:  "Code linting",
    7:  "Data serialisation & validation",
    8:  "Test runners",
    9:  "JS transpilation (Babel)",
    10: "PHP/Symfony ecosystem",
    11: "CLI tools & Ember addons",
    12: "CSS & styling",
    13: "Command-line infrastructure",
    14: "Task runners (Gulp/Rake)",
    15: "Web & mobile frameworks",
    16: "Parsers & code transformation",
    17: "Testing & mocking",
    18: "HTTP & networking",
    19: "Angular ecosystem",
    20: "Server infrastructure",
    21: "Compilation & code transforms",
    22: "Vue ecosystem",
    23: "Documentation & formatting",
    24: "General-purpose infrastructure (residual)",
}

FTES_PER_PACKAGE = 2
EU_MEDIAN_DEV_SALARY = 45_000

# Domain-specific stop words for package descriptions
DOMAIN_STOP_WORDS = [
    "package", "packages", "library", "libraries", "module", "modules",
    "plugin", "plugins", "component", "components", "extension", "extensions",
    "wrapper", "wrappers", "binding", "bindings", "adapter", "adapters",
    "middleware", "helper", "helpers", "utility", "utilities", "util", "utils",
    "toolkit", "toolset", "sdk", "api",
    "npm", "node", "nodejs", "js", "javascript",
    "php", "python", "ruby", "java", "net", "asp", "dotnet", "csharp",
    "cargo", "crate", "gem", "pip", "nuget", "packagist", "maven",
    "simple", "lightweight", "fast", "easy", "small", "tiny", "minimal",
    "modern", "powerful", "flexible", "robust", "provides", "support",
    "supports", "based", "using", "used", "use", "set", "just",
    "code", "project", "projects", "application", "applications", "app",
    "tool", "tools", "implementation", "implementations",
    "core", "base", "common", "standard", "default", "official",
    "version", "release", "latest", "new", "updated",
    "file", "files", "data", "type", "types", "entry", "stub",
    "method", "methods", "function", "functions", "class", "classes",
    "object", "objects", "interface", "interfaces",
    "config", "configuration", "settings", "options", "exported",
    "string", "strings", "stream", "streams", "buffer", "buffers",
    "browser", "convert", "build", "builds", "building",
    "run", "running", "create", "creating",
    "load", "loading", "read", "write", "render", "rendering",
    "native", "apps", "development", "definitions", "definition",
    "async", "callback", "promise", "event", "events",
    "add", "added", "adding", "install", "require",
    "cross", "platform", "like", "runtime", "directory",
]

def normalise(s):
    smin, smax = s.min(), s.max()
    if smax == smin:
        return pd.Series(0.5, index=s.index)
    return (s - smin) / (smax - smin)

def clean_descriptions(texts):
    cleaned = texts.copy()
    cleaned = cleaned.str.replace(r'https?://\S+', '', regex=True)
    cleaned = cleaned.str.replace(r'!\[.*?\]\(.*?\)', '', regex=True)
    cleaned = cleaned.str.replace(r'\[([^\]]*)\]\([^\)]*\)', r'\1', regex=True)
    cleaned = cleaned.str.replace(r'<[^>]+>', '', regex=True)
    return cleaned

stop_words = list(ENGLISH_STOP_WORDS) + DOMAIN_STOP_WORDS

## Stage 1: Load and Clean Libraries.io Data

In [ ]:
# Load the analysis-ready CSV produced by preprocess.py.
# Filtering, the CSV-header off-by-one fix, and column renaming are all
# done in preprocess.py; this notebook never touches the 25 GB raw file.
assert os.path.exists(PACKAGES_FILE), \
    f"{PACKAGES_FILE} not found. Run `python preprocess.py` first."

df = pd.read_csv(PACKAGES_FILE, low_memory=False)
print(f"Load-bearing packages: {len(df):,}")
print(f"Descriptions: {df['Description'].notna().sum():,} ({df['Description'].notna().mean()*100:.1f}%)")
df["Platform"].value_counts()


## Stage 2: Criticality and Fragility Scoring

In [ ]:
c1 = normalise(np.log1p(df["dep_pkg_count"]))
c2 = normalise(np.log1p(df["dep_repo_count"]))
df["criticality"] = normalise(W_DEP_PKG * c1 + W_DEP_REPO * c2)

df["contributors"] = df["contributors"].fillna(1).clip(lower=1)
f1 = normalise(1.0 / df["contributors"])

for col in ["last_release", "last_pushed"]:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)
best_date = df["last_release"].dt.tz_localize(None).fillna(df["last_pushed"].dt.tz_localize(None))
df["days_since_release"] = (SNAPSHOT_DATE - best_date).dt.days
df["days_since_release"] = df["days_since_release"].fillna(df["days_since_release"].quantile(0.9)).clip(lower=0)
f2 = normalise(np.log1p(df["days_since_release"]))

df["open_issues"] = df["open_issues"].fillna(0)
df["stars"] = df["stars"].fillna(0)
df["issue_ratio"] = df["open_issues"] / df["stars"].clip(lower=1)
f3 = normalise(df["issue_ratio"])

df["bus_factor"] = (df["contributors"] < 2).astype(float)
f4 = normalise(df["bus_factor"])

df["fragility"] = normalise(W_CONTRIBUTOR*f1 + W_STALENESS*f2 + W_ISSUES*f3 + W_BUSFACTOR*f4)

c_thresh = df["criticality"].quantile(RISK_PERCENTILE / 100)
f_thresh = df["fragility"].quantile(RISK_PERCENTILE / 100)
df["in_risk_universe"] = (df["criticality"] >= c_thresh) & (df["fragility"] >= f_thresh)

print(f"Thresholds: C>={c_thresh:.3f}, F>={f_thresh:.3f}")
print(f"Risk universe: {df['in_risk_universe'].sum()} packages")
df[df["in_risk_universe"]]["Platform"].value_counts()

## Stage 2b: Criticality-Fragility Correlation

In [ ]:
pearson_r, pearson_p = stats.pearsonr(df["criticality"], df["fragility"])
spearman_r, spearman_p = stats.spearmanr(df["criticality"], df["fragility"])

print(f"Pearson  r = {pearson_r:.4f}  (p = {pearson_p:.2e})")
print(f"Spearman \u03c1 = {spearman_r:.4f}  (p = {spearman_p:.2e})")

hi_c_hi_f = ((df["criticality"]>=c_thresh)&(df["fragility"]>=f_thresh)).sum()
expected = len(df) * 0.0625
print(f"\nRisk universe: {hi_c_hi_f} ({hi_c_hi_f/len(df)*100:.1f}%) vs {expected:.0f} expected if independent (6.25%)")
print(f"\nC and F are negatively correlated: more critical packages tend to be less fragile.")
print(f"The market partially works. The {hi_c_hi_f} risk-universe packages are the exceptions.")

## Stage 3a: Topic Count Selection

In [ ]:
texts = clean_descriptions(df["Description"].fillna("").astype(str))

tfidf = TfidfVectorizer(max_features=5000, stop_words=stop_words, min_df=5, max_df=0.7,
                         token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b")
tfidf_matrix = tfidf.fit_transform(texts)

k_range = range(5, 26)
errors = []
for k in k_range:
    nmf_test = NMF(n_components=k, random_state=RANDOM_SEED, max_iter=500)
    nmf_test.fit_transform(tfidf_matrix)
    errors.append(nmf_test.reconstruction_err_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), errors, "o-", color="steelblue", linewidth=2, markersize=5)
ax.axvline(N_TOPICS, color="crimson", linestyle="--", alpha=0.7, label=f"Selected k={N_TOPICS}")
ax.set_xlabel("Number of Topics (k)"); ax.set_ylabel("Reconstruction Error")
ax.set_title("NMF Topic Count Selection"); ax.legend(); ax.grid(True, alpha=0.3)
fig.text(0.5, -0.02,
         f"TF-IDF over N={tfidf_matrix.shape[0]:,} package descriptions; "
         f"NMF random_state={RANDOM_SEED}, max_iter=500. "
         f"k=25 selected by topic-quality inspection (no sharp elbow in the curve).",
         ha="center", fontsize=8, style="italic", wrap=True)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig0_topic_selection.png"), dpi=150, bbox_inches="tight"); plt.show()

## Stage 3b: Topic Modelling (NMF, k=25)

In [ ]:
feature_names = tfidf.get_feature_names_out()
nmf = NMF(n_components=N_TOPICS, random_state=RANDOM_SEED, max_iter=500)
W = nmf.fit_transform(tfidf_matrix)
H = nmf.components_
print(f"Reconstruction error: {nmf.reconstruction_err_:.2f}, iterations: {nmf.n_iter_}")

df["topic_id"] = W.argmax(axis=1)
df["topic_weight"] = W.max(axis=1)

topic_labels = {}
for tid in range(N_TOPICS):
    top_words = [feature_names[i] for i in H[tid].argsort()[-8:][::-1]]
    topic_labels[tid] = ", ".join(top_words[:4])
    n_pkgs = (df["topic_id"]==tid).sum()
    n_risk = ((df["topic_id"]==tid)&df["in_risk_universe"]).sum()
    print(f"T{tid:2d}: [{topic_labels[tid]:<40}] {n_pkgs:>4} pkgs, {n_risk:>2} risk")

# Apply human-readable labels
for tid, human_label in HUMAN_TOPIC_LABELS.items():
    if tid in topic_labels:
        topic_labels[tid] = human_label

df["topic_label"] = df["topic_id"].map(topic_labels)

print(f"\nHuman-readable labels applied:")
for tid in sorted(topic_labels.keys()):
    n = (df["topic_id"]==tid).sum()
    print(f"  T{tid:2d}: {topic_labels[tid]:<45} ({n} pkgs)")
print(f"\nLargest cluster: {df['topic_id'].value_counts().iloc[0]} ({df['topic_id'].value_counts().iloc[0]/len(df)*100:.1f}%)")

## Stage 4: Fragility Analysis by Cluster

In [ ]:
summary_rows = []
for tid in sorted(topic_labels.keys()):
    cluster = df[df["topic_id"]==tid]
    risk = cluster[cluster["in_risk_universe"]]
    summary_rows.append({
        "topic_id": tid, "label": topic_labels[tid],
        "n_packages": len(cluster), "n_risk": len(risk),
        "risk_pct": len(risk)/len(cluster)*100 if len(cluster)>0 else 0,
        "mean_fragility": cluster["fragility"].mean(),
        "median_fragility": cluster["fragility"].median(),
        "mean_criticality": cluster["criticality"].mean(),
        "pct_bus_factor_1": (cluster["contributors"]<2).mean()*100,
        "mean_days_since_release": cluster["days_since_release"].mean(),
    })
summary = pd.DataFrame(summary_rows).sort_values("mean_fragility", ascending=False)
summary[["topic_id","label","n_packages","n_risk","risk_pct","mean_fragility","mean_criticality","pct_bus_factor_1"]].round(3)

## Stage 4b: Inspecting the Most Fragile Cluster and Notable Risk Packages

In [ ]:
top_tid = int(summary.iloc[0]["topic_id"])
cluster = df[df["topic_id"]==top_tid]
print(f"Topic {top_tid} [{topic_labels[top_tid]}]: {len(cluster)} pkgs, "
      f"{(cluster['contributors']<2).mean()*100:.1f}% bus-factor=1\n")
print("Top 15 by criticality:")
for _, r in cluster.nlargest(15,"criticality")[["Name","Platform","criticality","fragility","contributors","dep_pkg_count","in_risk_universe"]].iterrows():
    flag = " *** RISK" if r["in_risk_universe"] else ""
    print(f"  {r['Name']:<40} [{r['Platform']:<10}] C={r['criticality']:.3f} F={r['fragility']:.3f} deps={int(r['dep_pkg_count']):>6}{flag}")

print(f"\nTop 15 risk-universe packages (all clusters):")
for _, r in df[df["in_risk_universe"]].nlargest(15,"criticality").iterrows():
    t = topic_labels[r["topic_id"]].split(",")[0].strip()
    print(f"  {r['Name']:<40} [{r['Platform']:<10}] C={r['criticality']:.3f} F={r['fragility']:.3f} topic={t}")

## Stage 4c: LASSO Validation of Fragility Weights

In [ ]:
target = (df["contributors"] < 2).astype(int)
features = pd.DataFrame({
    "log_days_since_release": np.log1p(df["days_since_release"]),
    "issue_ratio": df["issue_ratio"],
    "log_dep_pkg": np.log1p(df["dep_pkg_count"]),
    "log_dep_repo": np.log1p(df["dep_repo_count"]),
    "log_stars": np.log1p(df["stars"]),
    "log_open_issues": np.log1p(df["open_issues"]),
})

valid = features.notna().all(axis=1) & target.notna()
X = StandardScaler().fit_transform(features[valid].values)
y = target[valid].values

lasso = LassoCV(cv=5, random_state=RANDOM_SEED, max_iter=5000)
lasso.fit(X, y)
cv_scores = cross_val_score(lasso, X, y, cv=5, scoring="r2")

print(f"Optimal alpha: {lasso.alpha_:.6f}")
print(f"R\u00b2 in-sample: {lasso.score(X, y):.4f}, R\u00b2 5-fold CV: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
print(f"\nLASSO coefficients (standardised):")
coef_df = pd.DataFrame({"feature": features.columns, "coefficient": lasso.coef_})
coef_df = coef_df.reindex(coef_df["coefficient"].abs().sort_values(ascending=False).index)
for _, r in coef_df.iterrows():
    print(f"  {r['feature']:<30} {r['coefficient']:>8.4f}")

## Stage 4d: Threshold Sensitivity (60th, 75th, 90th Percentile)

In [ ]:
for pct in [60, 75, 90]:
    c_t = df["criticality"].quantile(pct/100)
    f_t = df["fragility"].quantile(pct/100)
    risk_s = df[(df["criticality"]>=c_t)&(df["fragility"]>=f_t)]
    cluster_risk = []
    for tid in sorted(topic_labels.keys()):
        cl = df[df["topic_id"]==tid]
        nr = len(risk_s[risk_s["topic_id"]==tid])
        cluster_risk.append({"label":topic_labels[tid].split(",")[0].strip(),
                             "risk_pct":nr/len(cl)*100 if len(cl)>0 else 0, "n_risk":nr})
    cr = pd.DataFrame(cluster_risk).sort_values("risk_pct",ascending=False)
    print(f"\n{pct}th pctile: {len(risk_s)} pkgs, EUR {len(risk_s)*FTES_PER_PACKAGE*EU_MEDIAN_DEV_SALARY/1e6:.1f}M gap")
    print(f"  Top 3: ", end="")
    print(", ".join(f"{r['label']} ({r['risk_pct']:.1f}%)" for _,r in cr.head(3).iterrows()))

## Stage 5: EU Software-Sector GVA from Eurostat

In [ ]:
# Eurostat GVA — cached by preprocess.py. No live network call here.
assert os.path.exists(EUROSTAT_FILE), \
    f"{EUROSTAT_FILE} not found. Run `python preprocess.py` first."

gva = pd.read_csv(EUROSTAT_FILE)
total_gva = gva["gva_meur"].sum()
print(f"Countries: {len(gva)}, Total J62/J63 GVA: EUR {total_gva:,.0f}M")
gva.sort_values("gva_meur", ascending=False).head(10)


## Stage 6: Funding Gap by Cluster

In [ ]:
cost_per_pkg = FTES_PER_PACKAGE * EU_MEDIAN_DEV_SALARY
risk = df[df["in_risk_universe"]].copy()
total_gap = len(risk) * cost_per_pkg
print(f"Risk universe: {len(risk)} pkgs x EUR {cost_per_pkg:,}/yr = EUR {total_gap/1e6:.1f}M")
print(f"EU software GVA: EUR {total_gva:,.0f}M\n")
gap_rows = []
for tid in sorted(topic_labels.keys()):
    nr = len(risk[risk["topic_id"]==tid])
    gap_rows.append({"topic_id":tid,"label":topic_labels[tid],"n_risk":nr,"gap_meur":nr*cost_per_pkg/1e6})
gap_df = pd.DataFrame(gap_rows).sort_values("gap_meur",ascending=False)
gap_df[gap_df["n_risk"]>0]

## Stage 7: Monte Carlo Bootstrap (1,000 Replicates)

In [ ]:
rng = np.random.default_rng(RANDOM_SEED)
n = len(df)
boot_results = []
for i in range(N_BOOTSTRAP):
    idx = rng.choice(n, size=n, replace=True)
    sample = df.iloc[idx]
    c_t = sample["criticality"].quantile(RISK_PERCENTILE/100)
    f_t = sample["fragility"].quantile(RISK_PERCENTILE/100)
    risk_b = sample[(sample["criticality"]>=c_t)&(sample["fragility"]>=f_t)]
    boot_results.append({"risk_size":len(risk_b),
        "mean_fragility":risk_b["fragility"].mean() if len(risk_b)>0 else 0,
        "funding_gap_meur":len(risk_b)*cost_per_pkg/1e6})
boot_df = pd.DataFrame(boot_results)
print(f"Point: {len(risk)} pkgs, EUR {len(risk)*cost_per_pkg/1e6:.1f}M\n95% CI:")
for col,label in [("risk_size","Risk size"),("funding_gap_meur","Gap (M EUR)"),("mean_fragility","Mean F (risk)")]:
    print(f"  {label}: [{boot_df[col].quantile(.025):.1f}, {boot_df[col].quantile(.975):.1f}] SE={boot_df[col].std():.2f}")

## Stage 8: Figures

### Figure 1: Criticality vs Fragility by Cluster

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
cmap = plt.colormaps.get_cmap("tab20b").resampled(N_TOPICS)
for tid in sorted(df["topic_id"].unique()):
    m = (df["topic_id"]==tid)&(~df["in_risk_universe"])
    ax.scatter(df.loc[m,"criticality"], df.loc[m,"fragility"], c=[cmap(tid)], alpha=0.15, s=8)
risk_df = df[df["in_risk_universe"]]
for tid in sorted(df["topic_id"].unique()):
    m = risk_df["topic_id"]==tid
    if m.sum()>0:
        ax.scatter(risk_df.loc[m,"criticality"], risk_df.loc[m,"fragility"], c=[cmap(tid)],
                   alpha=0.8, s=30, edgecolors="black", linewidths=0.5,
                   label=f"{topic_labels[tid].split(',')[0].strip()} ({m.sum()})")
ax.axvline(c_thresh, color="grey", linestyle="--", alpha=0.5)
ax.axhline(f_thresh, color="grey", linestyle="--", alpha=0.5)
ax.set_xlabel("Criticality"); ax.set_ylabel("Fragility")
ax.set_title("Criticality vs Fragility by Functional Cluster")
ax.legend(title="Risk universe", fontsize=6, title_fontsize=7, loc="upper left", ncol=3)
n_risk = int(df["in_risk_universe"].sum())
fig.text(0.5, -0.02,
         f"N={len(df):,} load-bearing packages (≥100 dependent projects). "
         f"Dashed grey lines mark the {RISK_PERCENTILE}th-percentile thresholds "
         f"(C≥{c_thresh:.3f}, F≥{f_thresh:.3f}). "
         f"Risk universe = {n_risk} packages above both thresholds, shown with "
         f"larger markers and black edges. Background points faded for legibility.",
         ha="center", fontsize=8, style="italic", wrap=True)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig1_scatter.png"), dpi=150, bbox_inches="tight"); plt.show()

### Figure 2: Cluster Fragility and Risk Concentration

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
cmap = plt.colormaps.get_cmap("tab20b").resampled(N_TOPICS)
s = summary.sort_values("mean_fragility", ascending=True)
axes[0].barh(range(len(s)), s["mean_fragility"], color=[cmap(t) for t in s["topic_id"]])
axes[0].set_yticks(range(len(s))); axes[0].set_yticklabels([l.split(",")[0].strip() for l in s["label"]], fontsize=7)
axes[0].set_xlabel("Mean Fragility"); axes[0].set_title("(a) Fragility by Cluster")
axes[0].axvline(df["fragility"].mean(), color="red", linestyle="--", alpha=0.7, label="Overall mean")
axes[0].legend(fontsize=8)
s2 = summary.sort_values("risk_pct", ascending=True)
axes[1].barh(range(len(s2)), s2["risk_pct"], color=[cmap(t) for t in s2["topic_id"]])
axes[1].set_yticks(range(len(s2))); axes[1].set_yticklabels([l.split(",")[0].strip() for l in s2["label"]], fontsize=7)
axes[1].set_xlabel("% in Risk Universe"); axes[1].set_title("(b) Risk Concentration")
n_risk = int(df["in_risk_universe"].sum())
fig.text(0.5, -0.03,
         f"N={len(df):,} load-bearing packages assigned to {N_TOPICS} NMF clusters. "
         f"Risk universe = {n_risk} packages above the {RISK_PERCENTILE}th percentile "
         f"on both criticality and fragility. Cluster labels are human-assigned "
         f"after inspecting top NMF keywords and member packages; Topics 0 and 24 are "
         f"residual clusters of generic-description packages.",
         ha="center", fontsize=8, style="italic", wrap=True)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig2_clusters.png"), dpi=150, bbox_inches="tight"); plt.show()

### Figure 3: Bootstrap Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax,(col,title,xl) in zip(axes,[("risk_size","Risk Universe Size","Packages"),
    ("funding_gap_meur","Funding Gap","M EUR"),("mean_fragility","Mean Fragility (Risk)","Score")]):
    ax.hist(boot_df[col], bins=40, color="steelblue", alpha=0.7, edgecolor="white")
    lo,hi = boot_df[col].quantile(.025), boot_df[col].quantile(.975)
    ax.axvline(lo, color="orange", linestyle="--", lw=1.5, label="95% CI")
    ax.axvline(hi, color="orange", linestyle="--", lw=1.5)
    ax.set_title(title); ax.set_xlabel(xl); ax.set_ylabel("Freq"); ax.legend(fontsize=8)
fig.suptitle(f"Monte Carlo Bootstrap ({N_BOOTSTRAP} replicates)", fontsize=13, fontweight="bold")
fig.text(0.5, -0.04,
         f"Resampling N={len(df):,} load-bearing packages with replacement, "
         f"recomputing the {RISK_PERCENTILE}th-percentile risk thresholds and the "
         f"funding-gap arithmetic on each replicate. Orange dashed lines mark the "
         f"percentile-based 95% CI (Efron & Tibshirani, 1993). "
         f"Funding gap uses 2 FTE × EUR 45,000/yr per risk package — a gross lower bound.",
         ha="center", fontsize=8, style="italic", wrap=True)
plt.tight_layout(); fig.savefig(os.path.join(OUTPUT_DIR, "fig3_bootstrap.png"), dpi=150, bbox_inches="tight"); plt.show()

## Stage 9: Save Outputs

In [ ]:
out_cols = ["ID","Platform","Name","Description","dep_pkg_count","dep_repo_count",
    "contributors","open_issues","stars","days_since_release","criticality","fragility",
    "in_risk_universe","topic_id","topic_label"]
df[out_cols].to_csv(os.path.join(OUTPUT_DIR, "packages_scored.csv"), index=False)
summary.to_csv(os.path.join(OUTPUT_DIR, "cluster_summary.csv"), index=False)
gap_df.to_csv(os.path.join(OUTPUT_DIR, "funding_gap_by_cluster.csv"), index=False)
print(f"Saved: packages_scored.csv ({len(df)} rows), cluster_summary.csv, funding_gap_by_cluster.csv")